# 高级流式

来源:
- https://docs.langchain.com/mcp/frontend/join-rejoin
- https://docs.langchain.com/mcp/frontend/time-travel
- https://docs.langchain.com/mcp/frontend/generative-ui

本笔记覆盖高级流式交互的三个主题：连接/重连(Join/Rejoin)、时间旅行(Time Travel)和生成式UI(Generative UI)。

---
## 1. 连接与重连 (Join / Rejoin)

Join/Rejoin 模式允许用户加入一个正在进行的对话流，或在断线后重新连接。

### 场景

```
场景A: 用户A开启了对话，用户B加入
场景B: 用户断线后重新连接到已有线程
场景C: 多用户协作同一Agent会话
```

### 架构设计

```
[用户A] ──→ [LangGraph Server] ──→ [Agent]
              ↕ SSE Stream
[用户B] ──→ Join 同一 Thread
              ↕ SSE Stream
[用户C] ──→ Rejoin (断线重连)
              ↓ 读取历史消息
              ↓ 恢复流式监听
```

### React 实现

```tsx
// TypeScript React
import { useStream } from "@langchain/langgraph-sdk/react";
import { useEffect, useState } from "react";

// 可共享/重连的聊天Hook
function useSharableChat(threadId?: string) {
  const [isReconnecting, setIsReconnecting] = useState(false);

  const stream = useStream({
    apiUrl: "http://localhost:8123",
    assistantId: "my_agent",
    threadId,  // 指定已有线程ID
    onConnect: () => {
      setIsReconnecting(false);
      console.log("连接成功");
    },
    onDisconnect: () => {
      console.log("连接断开，尝试重连...");
      setIsReconnecting(true);
    },
    onError: (error) => {
      console.error("连接错误:", error);
      // 自动重连逻辑
      setTimeout(() => {
        stream.reconnect();
      }, 3000);
    },
  });

  return { ...stream, isReconnecting };
}

// 加入现有会话
function JoinSession({ threadId }: { threadId: string }) {
  const { messages, isReconnecting, isLoading } = useSharableChat(threadId);

  return (
    <div className="joined-chat">
      {isReconnecting && (
        <div className="reconnect-banner">
          🔄 重连中...
        </div>
      )}
      
      <div className="session-info">
        <span>会话: {threadId}</span>
        <button onClick={() => navigator.clipboard.writeText(threadId)}>
          复制会话ID
        </button>
      </div>

      <MessageList messages={messages} />
      
      {isLoading && <TypingIndicator />}
      <MessageInput />
    </div>
  );
}

// 会话列表 (用于选择加入)
function SessionList() {
  const [sessions, setSessions] = useState<Array<{
    threadId: string;
    title: string;
    activeUsers: number;
  }>>([]);

  useEffect(() => {
    // 从服务端获取活跃会话列表
    fetchActiveSessions().then(setSessions);
  }, []);

  return (
    <div className="session-list">
      <h3>活跃会话</h3>
      {sessions.map(s => (
        <div key={s.threadId} className="session-item">
          <span>{s.title}</span>
          <span>{s.activeUsers} 人在线</span>
          <button onClick={() => joinSession(s.threadId)}>
            加入
          </button>
        </div>
      ))}
    </div>
  );
}
```

### 服务端重连支持

```python
# Python 服务端代码
from langgraph.checkpoint import MemorySaver

# 启用持久化检查点
checkpointer = MemorySaver()  # 或 PostgresSaver / SqliteSaver

graph = builder.compile(checkpointer=checkpointer)

# 重连时获取历史状态
def get_thread_history(thread_id: str):
    state = graph.get_state({
        "configurable": {"thread_id": thread_id}
    })
    return state

# 继续已有线程
def continue_thread(thread_id: str, new_input: dict):
    config = {"configurable": {"thread_id": thread_id}}
    for event in graph.stream(new_input, config):
        yield event
```

---
## 2. 时间旅行 (Time Travel)

时间旅行允许用户回溯到对话历史的任意状态点，从该点重新执行。

### 概念

```
[当前时间线]
  1 → 2 → 3 → 4 → 5
               │
               ▼ 回溯到状态3
  1 → 2 → 3 → 4' → 5' → 6' (新时间线)
```

### 关键机制

| 机制 | 说明 |
|------|------|
| **状态快照** | 每个节点执行后保存完整状态 |
| **检查点** | 可恢复的执行断点 |
| **分支** | 从历史状态创建新分支 |
| **回放** | 重新执行从某状态开始的路径 |

### React 实现

```tsx
// TypeScript React
import { useState, useCallback } from "react";
import { useStream } from "@langchain/langgraph-sdk/react";

interface Checkpoint {
  checkpointId: string;
  timestamp: string;
  node: string;
  state: Record<string, any>;
  parentCheckpointId: string | null;
}

function useTimeTravel(threadId: string) {
  const [checkpoints, setCheckpoints] = useState<Checkpoint[]>([]);
  const [activeCheckpoint, setActiveCheckpoint] = useState<string | null>(null);
  const stream = useStream({ apiUrl, assistantId, threadId });

  // 加载历史检查点
  const loadCheckpoints = useCallback(async () => {
    const history = await fetchCheckpoints(threadId);
    setCheckpoints(history);
  }, [threadId]);

  // 回溯到指定检查点
  const travelTo = useCallback(async (checkpointId: string) => {
    setActiveCheckpoint(checkpointId);
    const state = await getCheckpointState(threadId, checkpointId);
    stream.restoreState(state);
  }, [threadId, stream]);

  // 从检查点重新执行
  const replayFrom = useCallback(async (checkpointId: string) => {
    setActiveCheckpoint(checkpointId);
    await stream.resume({ checkpointId });
  }, [stream]);

  return {
    checkpoints,
    activeCheckpoint,
    loadCheckpoints,
    travelTo,
    replayFrom,
    currentState: stream.messages,
  };
}

// 时间旅行UI
function TimeTravelPanel({ threadId }: { threadId: string }) {
  const {
    checkpoints,
    activeCheckpoint,
    loadCheckpoints,
    travelTo,
    replayFrom,
  } = useTimeTravel(threadId);

  useEffect(() => {
    loadCheckpoints();
  }, [loadCheckpoints]);

  return (
    <div className="time-travel-panel">
      <h3>⏱ 时间旅行</h3>
      
      {/* 时间线滑块 */}
      <div className="timeline">
        <input
          type="range"
          min={0}
          max={checkpoints.length - 1}
          value={checkpoints.findIndex(c => c.checkpointId === activeCheckpoint)}
          onChange={(e) => {
            const cp = checkpoints[Number(e.target.value)];
            if (cp) travelTo(cp.checkpointId);
          }}
        />
      </div>

      {/* 检查点列表 */}
      <div className="checkpoint-list">
        {checkpoints.map((cp, i) => (
          <div
            key={cp.checkpointId}
            className={`checkpoint-item ${
              cp.checkpointId === activeCheckpoint ? "active" : ""
            }`}
          >
            <div className="cp-header">
              <span>步骤 {i}</span>
              <span className="cp-node">{cp.node}</span>
            </div>
            <div className="cp-time">
              {new Date(cp.timestamp).toLocaleTimeString()}
            </div>
            <div className="cp-actions">
              <button onClick={() => travelTo(cp.checkpointId)}>
                查看
              </button>
              <button onClick={() => replayFrom(cp.checkpointId)}>
                从此重放
              </button>
            </div>
          </div>
        ))}
      </div>
    </div>
  );
}
```

---
## 3. 生成式 UI (Generative UI)

生成式UI允许Agent动态生成React组件，实现超越文本的富交互体验。

### 架构

```
[Agent]
    │
    ├── 文本输出 → Markdown渲染
    ├── 工具调用 → ToolCallCard
    └── UI组件描述 → [组件渲染器] → 动态React组件
         (JSON schema)         │
                               ├── <Chart />
                               ├── <Form />
                               ├── <Table />
                               └── <CustomComponent />
```

### 组件描述格式

```typescript
// TypeScript
interface GenerativeUIComponent {
  type: "chart" | "form" | "table" | "card" | "custom";
  props: Record<string, any>;
  children?: GenerativeUIComponent[];
  events?: {
    onClick?: string;  // 动作名称
    onSubmit?: string;
    onChange?: string;
  };
}

// Agent生成的UI描述示例
const agentGeneratedUI: GenerativeUIComponent = {
  type: "card",
  props: { title: "销售数据", variant: "elevated" },
  children: [
    {
      type: "chart",
      props: {
        type: "bar",
        data: [
          { month: "一月", sales: 12000 },
          { month: "二月", sales: 15000 },
          { month: "三月", sales: 11000 },
        ],
        xAxis: "month",
        yAxis: "sales",
      },
    },
    {
      type: "table",
      props: {
        columns: ["产品", "销量", "增长"],
        rows: [
          ["产品A", 500, "+12%"],
          ["产品B", 320, "-5%"],
        ],
      },
    },
    {
      type: "form",
      props: {
        fields: [
          { name: "dateRange", label: "日期范围", type: "date" },
          { name: "category", label: "类别", type: "select",
            options: ["全部", "电子产品", "服装"] },
        ],
        submitLabel: "重新查询",
        onSubmit: "refreshData"
      },
    },
  ],
};
```

### React 组件渲染器

```tsx
// TypeScript React
import React from "react";
import { BarChart, LineChart, PieChart } from "recharts";

// 组件注册表
const componentRegistry: Record<string, React.ComponentType<any>> = {
  "chart": ChartRenderer,
  "table": TableRenderer,
  "form": FormRenderer,
  "card": CardRenderer,
};

// 递归渲染器
function UIRenderer({ component, onEvent }: {
  component: GenerativeUIComponent;
  onEvent: (event: string, data: any) => void;
}) {
  const Component = componentRegistry[component.type];
  if (!Component) return <div>未知组件: {component.type}</div>;

  return (
    <Component
      {...component.props}
      events={component.events}
      onEvent={onEvent}
    >
      {component.children?.map((child, i) => (
        <UIRenderer key={i} component={child} onEvent={onEvent} />
      ))}
    </Component>
  );
}

// 图表渲染器
function ChartRenderer({ type, data, xAxis, yAxis }: any) {
  switch (type) {
    case "bar":
      return <BarChart width={500} height={300} data={data}>
        <XAxis dataKey={xAxis} />
        <YAxis />
        <Bar dataKey={yAxis} fill="#8884d8" />
      </BarChart>;
    case "line":
      return <LineChart width={500} height={300} data={data}>
        <Line type="monotone" dataKey={yAxis} />
      </LineChart>;
    default:
      return <pre>{JSON.stringify(data, null, 2)}</pre>;
  }
}

// 集成到消息流
function MessageWithGenerativeUI({ msg }: { msg: any }) {
  const handleEvent = async (event: string, data: any) => {
    // 将UI事件发送回Agent
    await submit({
      messages: [{
        role: "human",
        content: JSON.stringify({ event, data })
      }]
    });
  };

  return (
    <div className="message-with-ui">
      {msg.content && <MarkdownMessage content={msg.content} />}
      {msg.generativeUI && (
        <UIRenderer
          component={msg.generativeUI}
          onEvent={handleEvent}
        />
      )}
    </div>
  );
}
```

### CopilotKit 中的生成式UI

```tsx
// TypeScript React
import { useCopilotAction } from "@copilotkit/react-core";

function DataDashboard() {
  // Agent可以调用此动作动态渲染UI
  useCopilotAction({
    name: "renderDashboard",
    description: "渲染数据仪表盘",
    parameters: [
      {
        name: "chartType",
        type: "string",
        enum: ["bar", "line", "pie"],
      },
      {
        name: "data",
        type: "object",
        description: "图表数据",
      },
    ],
    render: ({ args }) => {
      return <ChartRenderer type={args.chartType} data={args.data} />;
    },
  });

  return <div>仪表盘就绪</div>;
}
```

### assistant-ui 中的生成式UI

```tsx
// TypeScript React
import { useAssistantRuntime } from "@assistant-ui/react";

// 注册自定义渲染器
const runtime = useLangGraphRuntime({
  apiUrl: "http://localhost:8123",
  assistantId: "generative_ui_agent",
  customRenderers: {
    chart: (props) => <ChartRenderer {...props} />,
    form: (props) => <FormRenderer {...props} />,
    custom: MyCustomComponent,
  },
});
```

---
## 4. 高级流式模式对比

| 模式 | 描述 | 适用场景 | 技术要求 |
|------|------|----------|----------|
| **Join/Rejoin** | 多用户加入/断线重连 | 协作工具、客服 | SSE、状态持久化、检查点 |
| **Time Travel** | 回溯和重放历史状态 | 调试、回溯分析 | 完整检查点、状态恢复API |
| **Generative UI** | Agent动态生成UI组件 | 数据分析、表单 | 组件注册表、事件回传机制 |

## 5. 最佳实践

1. **连接管理**: 实现指数退避重连策略，避免频繁重试
2. **状态同步**: Join时先同步完整历史，再开始流式监听
3. **检查点策略**: 合理设置检查点频率，平衡存储和恢复精度
4. **组件安全**: Generative UI组件需沙箱化，防止XSS
5. **事件回传**: UI事件通过工具调用机制回传给Agent
6. **性能优化**: Time Travel大量检查点时使用虚拟列表

## 6. 参考链接

- [Join/Rejoin](https://docs.langchain.com/mcp/frontend/join-rejoin)
- [Time Travel](https://docs.langchain.com/mcp/frontend/time-travel)
- [Generative UI](https://docs.langchain.com/mcp/frontend/generative-ui)
- [LangGraph 检查点文档](https://langchain-ai.github.io/langgraph/concepts/persistence/)
- [LangGraph SDK React](https://www.npmjs.com/package/@langchain/langgraph-sdk)
- [Recharts](https://recharts.org/)
- [CopilotKit 文档](https://docs.copilotkit.ai/)